# [실습 8] KoAlpaca-v1.1a 로 모방학습(SFT)

**목표** — 허깅페이스 허브의 한국어 지시-응답 데이터셋 [`Beomi/KoAlpaca-v1.1a`](https://huggingface.co/datasets/Beomi/KoAlpaca-v1.1a) 를 그대로 사용해, 작은 LLM을 **모방학습(Supervised Fine-Tuning)** 으로 한국어 비서 톤에 맞춥니다.

이 과정은 강화학습에서의 **모방학습(Behavioral Cloning)** 에 해당합니다.

- 정책 $\pi_\theta(a|s)$ → 다음 토큰 분포
- 시범 데이터 $(s, a^*)$ → `(instruction, output)` 한 쌍
- 손실 $\mathcal{L} = -\log \pi_\theta(a^*|s)$ → Causal LM cross-entropy

| 항목 | 값 |
|---|---|
| 데이터셋 | `Beomi/KoAlpaca-v1.1a` (≈ 21k 한국어 지시-응답 쌍) |
| 베이스 모델 | `Qwen/Qwen2.5-0.5B-Instruct` |
| 학습 방식 | QLoRA (4-bit 양자화 + LoRA 어댑터) |
| 권장 환경 | Google Colab T4 (16GB VRAM) |

## 0. Colab 환경 준비

런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 진행.

In [ ]:
!pip install -q --upgrade \
    transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 \
    peft==0.12.0 trl==0.10.1 bitsandbytes==0.43.3 sentencepiece

In [ ]:
import os, torch
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print("torch     :", torch.__version__)
print("cuda 사용 :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU       :", torch.cuda.get_device_name(0))

---
## 1. 데이터셋 로드 & 탐색 — `Beomi/KoAlpaca-v1.1a`

Stanford Alpaca를 한국어로 옮겨 정제한 데이터셋입니다.  
컬럼은 `instruction` (사용자 질문)과 `output` (모범 답변) 두 개만 있는 단순한 구조입니다.

In [ ]:
from datasets import load_dataset

ds = load_dataset("Beomi/KoAlpaca-v1.1a", split="train")
print("총 샘플 수:", len(ds))
print("컬럼      :", ds.column_names)
print("\n첫 번째 샘플\n" + "-" * 60)
print("instruction:", ds[0]["instruction"])
print("output     :", ds[0]["output"])

In [ ]:
# 길이 분포 살펴보기 — max_seq_length 잡을 때 근거가 됨
import numpy as np

lens_inst = np.array([len(x) for x in ds["instruction"]])
lens_out  = np.array([len(x) for x in ds["output"]])

def stats(name, arr):
    print(f"{name:>11s}  mean={arr.mean():6.1f}  p50={np.percentile(arr,50):6.0f}  "
          f"p95={np.percentile(arr,95):6.0f}  max={arr.max():6.0f}")

stats("instruction", lens_inst)
stats("output     ", lens_out)

In [ ]:
# 무작위로 3개 샘플 보기
import random
random.seed(0)
for i in random.sample(range(len(ds)), 3):
    print(f"[{i}] Q: {ds[i]['instruction']}")
    print(f"     A: {ds[i]['output'][:200]}{'...' if len(ds[i]['output'])>200 else ''}")
    print("-" * 60)

---
## 2. 프롬프트 포맷팅

모델이 "지시 → 응답" 패턴을 학습하도록 다음과 같이 가공합니다.

```
### 명령어:
{instruction}

### 응답:
{output}<eos>
```

`<eos>` 토큰을 붙여 "여기서 답이 끝난다" 는 신호를 명확히 줘야 무한 생성이 줄어듭니다.

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

PROMPT_TEMPLATE = (
    "### 명령어:\n{instruction}\n\n"
    "### 응답:\n{output}" + tokenizer.eos_token
)

def format_alpaca(example):
    # SFTTrainer는 batched=True 입력을 받으므로 list 단위로 반환
    texts = []
    for inst, out in zip(example["instruction"], example["output"]):
        texts.append(PROMPT_TEMPLATE.format(instruction=inst, output=out))
    return texts

# 미리 한 샘플 직접 찍어보기
sample_text = PROMPT_TEMPLATE.format(
    instruction=ds[0]["instruction"],
    output=ds[0]["output"],
)
print(sample_text[:400], "...\n")
print("토큰 개수:", len(tokenizer(sample_text)["input_ids"]))

---
## 3. 베이스 모델 로드 — QLoRA (4-bit)

T4 16GB 에서도 0.5B 모델을 안정적으로 돌리려고 **4-bit 양자화**를 씁니다.  
학습 가능한 파라미터는 LoRA 어댑터(약 0.5% 수준)뿐이라 메모리/시간 모두 크게 절약됩니다.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False   # gradient checkpointing 호환

print(model)

### 학습 전 응답 — "베이스라인"

학습 효과를 비교하려면 **학습 전** 응답을 먼저 찍어두는 게 좋습니다.

In [ ]:
EVAL_PROMPTS = [
    "점심 메뉴 추천해줘. 매운 건 못 먹어.",
    "파이썬에서 리스트를 역순으로 뒤집는 가장 효율적인 방법은?",
    "머신러닝에서 과적합을 막는 방법 세 가지만 말해줘.",
]

@torch.no_grad()
def generate(prompt, max_new_tokens=160):
    text = f"### 명령어:\n{prompt}\n\n### 응답:\n"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=0.7, top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("=" * 60)
print("[학습 전 베이스라인]")
print("=" * 60)
baseline = {}
for q in EVAL_PROMPTS:
    a = generate(q)
    baseline[q] = a
    print("Q:", q)
    print("A:", a)
    print("-" * 60)

---
## 4. LoRA 설정 & SFTTrainer 구성

- `r=8` : LoRA rank — 너무 작으면 표현력 부족, 너무 크면 메모리 증가
- `target_modules` : Attention의 Q/K/V/O 프로젝션 — Qwen 계열 기본 선택
- `max_steps=200` : 데모 — 실제 학습은 1~3 epoch 권장

In [ ]:
from peft import LoraConfig
from transformers import TrainingArguments
from trl import SFTTrainer

# 실습 속도를 위해 일부만 — 전체로 돌릴 거면 ds 그대로 사용
train_ds = ds.shuffle(seed=42).select(range(2000))
print("학습 샘플 수:", len(train_ds))

peft_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)

training_args = TrainingArguments(
    output_dir="./koalpaca_sft_out",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,    # 유효 배치 = 16
    learning_rate=2e-4,
    max_steps=200,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    group_by_length=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    peft_config=peft_config,
    formatting_func=format_alpaca,
    tokenizer=tokenizer,
    max_seq_length=768,
    args=training_args,
)

# 학습 가능한 파라미터 수 확인
trainer.model.print_trainable_parameters()

---
## 5. 학습 실행

T4 기준 200스텝 ≈ 8~12분 정도 소요됩니다.  
`loss` 가 처음에는 2~3 근처에서 시작해 1.0~1.5 부근까지 떨어지면 정상입니다.

In [ ]:
train_result = trainer.train()
print("\n학습 완료!")
print("최종 loss :", train_result.training_loss)
print("총 스텝수 :", train_result.global_step)

In [ ]:
# 어댑터 저장 — 추론 시 베이스 모델 + 어댑터만 결합하면 됨 (크기 ≈ 수십 MB)
SAVE_DIR = "./koalpaca_lora_adapter"
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

import os
print("\n저장 경로:", SAVE_DIR)
for f in os.listdir(SAVE_DIR):
    print(" -", f)

---
## 6. 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
steps = [h["step"] for h in history if "loss" in h]
losses = [h["loss"] for h in history if "loss" in h]

plt.figure(figsize=(8, 3))
plt.plot(steps, losses, marker="o", linewidth=1)
plt.title("KoAlpaca SFT — Training Loss")
plt.xlabel("step"); plt.ylabel("loss")
plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 7. 학습 후 응답 — 베이스라인과 비교

In [ ]:
print("=" * 60)
print("[학습 후 응답]")
print("=" * 60)
for q in EVAL_PROMPTS:
    a_after = generate(q)
    print("Q:", q)
    print("BEFORE:", baseline[q])
    print("AFTER :", a_after)
    print("-" * 60)

---
## 8. 저장된 어댑터로 추론하기 (재현용)

다음 세션에서 베이스 모델 + 어댑터만 결합해 추론하는 방법입니다.

In [ ]:
# from peft import PeftModel
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
#
# bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
#                          bnb_4bit_compute_dtype=torch.bfloat16)
# base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct",
#                                             quantization_config=bnb, device_map="auto")
# tok  = AutoTokenizer.from_pretrained("./koalpaca_lora_adapter")
# inf_model = PeftModel.from_pretrained(base, "./koalpaca_lora_adapter")
# inf_model.eval()
print("위 셀의 주석을 풀면 어댑터 가중치만 다시 붙여 즉시 추론 가능.")

---
## 9. 정리

- ✅ 실제 한국어 데이터셋 `Beomi/KoAlpaca-v1.1a` 를 그대로 사용
- ✅ QLoRA(4-bit + LoRA)로 16GB GPU 한 장에서도 학습 가능
- ✅ 학습 전/후 응답 비교로 모방학습의 효과를 정성적으로 확인

### 한 걸음 더
- 🔧 `max_steps` 를 500~1000 으로 늘려 학습 강도를 비교
- 🔧 `temperature`, `top_p` 를 바꿔가며 응답 다양성 vs 일관성 비교
- 🔧 다음 실습 [[실습9] Ko-Ultrafeedback DPO] 로 이어서 "좋은 답 vs 나쁜 답" 비교 학습 추가

### 데이터셋 라이선스
- KoAlpaca-v1.1a: CC-BY-NC 4.0 — 학습/연구 목적은 가능하나 상업적 사용 시 라이선스 확인 필요